In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Creating the dataframe
df  = pd.read_csv('german_credit_data.csv')
df

FileNotFoundError: [Errno 2] No such file or directory: 'german_credit_data.csv'

In [ ]:
df.shape

In [ ]:
df.sample(5)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# One-hot encoding for multicategorical data
# Binary encoding for two catgory
# Label encoding for the labels
df["Risk"].value_counts()

In [ ]:
# As the savings account and checking account has huge impact 
# We cant fill them 
df.drop(columns = 'Unnamed: 0', inplace=True)

In [ ]:
df.sample(5)

In [ ]:
df = df.dropna().reset_index(drop=True)

In [ ]:
df.shape

In [ ]:
# Plot the age, credit amount, duration
df[["Age", "Credit amount", "Duration"]].hist(bins=20, edgecolor='black')
plt.suptitle("Distribution of numerical features.", fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
for i, col in enumerate(["Age", "Duration", "Credit amount"]):
    plt.subplot(1, 3, i+1)
    sns.boxplot(y=df[col], color='skyblue')
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:
df.columns

In [ ]:
# Filtering the categorical columns
categorical_cols = ["Sex", "Job", "Housing", "Purpose", "Saving accounts", "Checking account"]

In [ ]:
plt.figure(figsize=(16,10))
for i, col in enumerate(categorical_cols):
    plt.subplot(3, 3, i+1)
    sns.countplot(data=df, x=col, palette= "Set2", order=df[col].value_counts().index)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)


plt.tight_layout()
plt.show()

In [ ]:
corr = df[["Age", "Job", "Credit amount", "Duration"]].corr()

In [ ]:
corr

In [ ]:
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.show()

In [ ]:
df.groupby("Job")["Credit amount"].mean()

In [ ]:
df.groupby("Sex")["Credit amount"].mean()

In [ ]:
pd.pivot_table(df, values="Credit amount", index="Housing", columns="Purpose")

In [ ]:
sns.scatterplot(data=df, x="Age", y="Credit amount", hue="Sex", size="Duration", alpha=0.7, palette="Set1")
plt.title("Credit Amount vs Age colored by Sex and sized by Duration")
plt.show()

In [ ]:
sns.violinplot(data=df, x="Saving accounts", y="Credit amount", palette="Pastel1", hue="Saving accounts", legend=False)
plt.title("Credit account distribution by Savings Account")
plt.show()

In [ ]:
df["Risk"].value_counts(normalize=True) * 100

In [ ]:
for i, col in enumerate(["Age", "Credit amount", "Duration"]):
    plt.subplot(1, 3, i+1)
    sns.boxplot(data=df, x="Risk", y=col, palette="Pastel2", hue="Risk", legend=False)
    plt.title(f"{col} by Risk")

plt.tight_layout()
plt.show()

In [ ]:
df.groupby("Risk")[["Age", "Credit amount", "Duration"]].mean()

In [ ]:
# Again we will plot the count plot for all the categorical data 
plt.figure(figsize=(13,10))
for i, col in enumerate(categorical_cols):
    plt.subplot(3, 3, i+1)
    sns.countplot(data=df, x=col, hue="Risk", palette="Set1", order=df[col].value_counts().index)
    plt.title(f"{col} by Risk")
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Feature engineering started 
df2 = df.drop(columns="Purpose", axis=1)

In [ ]:
df2.columns

In [ ]:
from sklearn.preprocessing import LabelEncoder
import joblib

In [ ]:
cat_cols = df2.select_dtypes(include="object").columns.drop("Risk")

In [ ]:
le_dict = {}

In [ ]:
for col in cat_cols:
    le = LabelEncoder()
    df2[col] = le.fit_transform(df2[col])
    le_dict[col] = le
    joblib.dump(le, f"{col}_encoder.pkl")

In [ ]:
le_target = LabelEncoder()
df2["Risk"] = le_target.fit_transform(df2["Risk"])

In [ ]:
df2["Risk"]

In [ ]:
joblib.dump(le_target, "Risk_encoder.pkl")

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df2.drop(columns="Risk")
y = df2["Risk"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

In [ ]:
# Function to train the model 
def train_model(model, param_grid, X_train, y_train, X_test, y_test):
    grid = GridSearchCV(model, param_grid, cv = 5, scoring="accuracy", n_jobs=-1)
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return best_model, acc, grid.best_params_

In [ ]:
dt = DecisionTreeClassifier(random_state=42, class_weight="balanced")
dt_param_grid = {
    "max_depth": [3,5,7,10,None],
    "min_samples_split": [2,5,10],
    "min_samples_leaf": [1,2,4]
}

In [ ]:
best_dt, acc_dt, params_dt = train_model(dt, dt_param_grid, X_train, y_train, X_test, y_test)

In [ ]:
print("Decision Tree accuracy:- ", acc_dt)

In [ ]:
print("Best parameters", params_dt)

In [ ]:
rf = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

In [ ]:
rf_params_grid = {
    "n_estimators":[100,200,300],
    "max_depth":[5,7,10,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}

In [ ]:
best_rf , acc_rf, best_params_rf = train_model(rf, rf_params_grid, X_train, y_train, X_test, y_test)

In [ ]:
print("Randome Forest accuracy:- ", acc_rf)

In [ ]:
print("Best parameters of random forest:- ", best_params_rf)

In [ ]:
et = ExtraTreesClassifier(class_weight="balanced", random_state=42, n_jobs=-1)
et_params_grid = {
    "n_estimators":[100,200,300],
    "max_depth":[5,7,10,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}

In [ ]:
best_et, acc_et, best_params_et = train_model(et, et_params_grid, X_train, y_train, X_test, y_test)

In [ ]:
print("Extra trees classifier accuracy:- ", acc_et)

In [ ]:
print("Extra trees best parameters:- ", best_params_et)

In [ ]:
 xgb = XGBClassifier(random_state=1, scale_pos_weight=(y_train == 0).sum()/ (y_train == 1).sum(), use_label_encoder=False, eval_metric="logloss")

In [ ]:
xgb_params_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.7, 1],
    "colsample_bytree": [0.7, 1]
}

In [ ]:
best_xgb, acc_xgb, best_params_xgb = train_model(xgb, xgb_params_grid, X_train, y_train, X_test, y_test)

In [ ]:
print("XGBoost accuracy:- ", acc_xgb)

In [ ]:
print("XGBoost best params:- ", best_params_xgb)

In [ ]:
joblib.dump(best_xgb, "xgb_credit_model.pkl")